In [1]:
!pip -q install fair-esm pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 5.9 MB/s eta 0:00:00


In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: True
Device: Tesla T4


In [5]:
from google.colab import files

uploaded = files.upload()
input_file = next(iter(uploaded))

print("Uploaded:", input_file)

Saving STEP8_60K_WITH_HEMOPI2_HC50.csv to STEP8_60K_WITH_HEMOPI2_HC50.csv
Uploaded: STEP8_60K_WITH_HEMOPI2_HC50.csv


In [6]:
import time
import json
import hashlib
import numpy as np
import pandas as pd
import torch
import esm

from pathlib import Path

# -----------------------------
# Paths
# -----------------------------
OUT_DIR = Path("/content/Step9_ESM2_Embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_NPZ = OUT_DIR / "STEP9_ESM2_35M_EMBEDDINGS_60K.npz"
META_CSV = OUT_DIR / "STEP9_60K_EMBEDDING_METADATA.csv"
MANIFEST = OUT_DIR / "STEP9_ESM2_manifest.json"

# -----------------------------
# Input
# -----------------------------
df = pd.read_csv(input_file, low_memory=False)

df["sequence"] = (
    df["sequence"]
    .astype(str)
    .str.strip()
    .str.upper()
)

assert len(df) == 60000
assert df["sequence"].nunique() == 60000

print("Rows:", len(df))
print("Unique sequences:", df["sequence"].nunique())

# -----------------------------
# GPU
# -----------------------------
assert torch.cuda.is_available()
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

# -----------------------------
# Load ESM-2
# -----------------------------
model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
model = model.eval().to(device)

batch_converter = alphabet.get_batch_converter()

REP_LAYER = 12
EMBED_DIM = 480
BATCH_SIZE = 256

embeddings = np.zeros(
    (len(df), EMBED_DIM),
    dtype=np.float16
)

# -----------------------------
# Run
# -----------------------------
times = []

for start in range(0, len(df), BATCH_SIZE):

    end = min(start + BATCH_SIZE, len(df))

    batch_df = df.iloc[start:end]

    batch_data = [
        (f"seq_{i}", seq)
        for i, seq in zip(
            batch_df.index,
            batch_df["sequence"]
        )
    ]

    _, strs, tokens = batch_converter(batch_data)

    tokens = tokens.to(device)

    t0 = time.time()

    with torch.no_grad():
        result = model(
            tokens,
            repr_layers=[REP_LAYER],
            return_contacts=False,
        )

    reps = result["representations"][REP_LAYER]

    vecs = []

    for i, seq in enumerate(strs):
        L = len(seq)

        vec = (
            reps[i, 1:L+1]
            .mean(dim=0)
            .cpu()
            .numpy()
        )

        vecs.append(vec)

    embeddings[start:end] = np.asarray(
        vecs,
        dtype=np.float16
    )

    elapsed = time.time() - t0
    times.append(elapsed)

    avg = np.mean(times)

    remaining_batches = (
        len(df) - end + BATCH_SIZE - 1
    ) // BATCH_SIZE

    eta = avg * remaining_batches

    print(
        f"{end:,}/60,000 | "
        f"batch {elapsed:.2f}s | "
        f"ETA {eta/60:.1f} min",
        flush=True
    )

# -----------------------------
# Save embeddings
# -----------------------------
np.savez_compressed(
    FINAL_NPZ,
    embeddings=embeddings,
    sequence=df["sequence"].to_numpy()
)

# -----------------------------
# Save metadata
# -----------------------------
meta = df.copy()

meta["esm2_embedding_row"] = np.arange(len(meta))
meta["esm2_model"] = "esm2_t12_35M_UR50D"
meta["esm2_representation_layer"] = 12
meta["esm2_pooling"] = "mean_residue_pooling"
meta["esm2_embedding_dim"] = 480

meta.to_csv(META_CSV, index=False)

# -----------------------------
# Manifest
# -----------------------------
manifest = {
    "rows": len(df),
    "unique_sequences": int(df["sequence"].nunique()),
    "model": "esm2_t12_35M_UR50D",
    "representation_layer": 12,
    "pooling": "mean_residue_pooling",
    "embedding_dimension": 480,
    "dtype": "float16",
    "batch_size": BATCH_SIZE,
    "device": torch.cuda.get_device_name(0),
}

MANIFEST.write_text(json.dumps(manifest, indent=2))

print("\nDONE")
print("Embeddings:", FINAL_NPZ)
print("Metadata:", META_CSV)
print("Manifest:", MANIFEST)

Rows: 60000
Unique sequences: 60000
GPU: Tesla T4
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t12_35M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t12_35M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t12_35M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t12_35M_UR50D-contact-regression.pt
256/60,000 | batch 0.84s | ETA 3.3 min
512/60,000 | batch 0.19s | ETA 2.0 min
768/60,000 | batch 0.19s | ETA 1.6 min
1,024/60,000 | batch 0.20s | ETA 1.4 min
1,280/60,000 | batch 0.20s | ETA 1.2 min
1,536/60,000 | batch 0.20s | ETA 1.2 min
1,792/60,000 | batch 0.19s | ETA 1.1 min
2,048/60,000 | batch 0.19s | ETA 1.0 min
2,304/60,000 | batch 0.19s | ETA 1.0 min
2,560/60,000 | batch 0.20s | ETA 1.0 min
2,816/60,000 | batch 0.19s | ETA 1.0 min
3,072/60,000 | batch 0.20s | ETA 0.9 min
3,328/60,000 | batch 0.20s | ETA 0.9 min
3,584/60,000 | batch 0.20s | ETA 0.9 min
3,840/60,000 | batch 0.19s | ETA 0.9 min
4,096/60,00